In [4]:
import os
from groq import Groq
from langsmith import Client

from langchain_huggingface import HuggingFaceEndpointEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Environment variables
HF_API_TOKEN = os.environ.get("HF_API_TOKEN")
QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")

# Model Configuration
HF_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GROQ_LLM_MODEL = "qwen/qwen3-32b"

print("="*60)
print("RAG PIPELINE CONFIGURATION")
print("="*60)
print(f"LLM Model (Groq): {GROQ_LLM_MODEL}")
print(f"Embedding Model (HuggingFace): {HF_EMBEDDING_MODEL}")
print(f"Vector DB URL: {QDRANT_URL}")
print("="*60)

c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAG PIPELINE CONFIGURATION
LLM Model (Groq): qwen/qwen3-32b
Embedding Model (HuggingFace): sentence-transformers/all-MiniLM-L6-v2
Vector DB URL: http://localhost:6333


In [6]:


model = "sentence-transformers/all-MiniLM-L6-v2"
hf = HuggingFaceEndpointEmbeddings(
            model=model,
            huggingfacehub_api_token=os.environ["HF_API_TOKEN"],
        )



### download an example reference data point from LANGSMITH

In [77]:
client=Client()


In [78]:
dataset=client.read_dataset(dataset_name="rag-evaluation-dataset")

In [79]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluation of RAG application.', data_type=<DataType.kv: 'kv'>, id=UUID('d9189034-6c37-45eb-a583-be410198ad12'), created_at=datetime.datetime(2026, 3, 29, 12, 59, 36, 334153, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 3, 29, 12, 59, 36, 334153, tzinfo=TzInfo(0)), example_count=28, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26200-SP0', 'sdk_version': '0.7.22', 'runtime_version': '3.12.13', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [80]:
list(client.list_examples(dataset_id=dataset.id,limit=10))[9].inputs

{'question': "What is the name of the album by John Mellencamp that features the song 'Orpheus Descending'?"}

In [81]:
list(client.list_examples(dataset_id=dataset.id,limit=10))[9].outputs

{'ground_truth': 'Orpheus Descending',
 'reference_context_ids': ['B0C2J9SYDQ'],
 'reference_descriptions': ['Orpheus Descending       Explicit Lyrics With a career spanning over 40 years, musician, artist and activist John Mellencamp is releasing his 25th full-length studio album, Orpheus Descending, via Republic Records. It was recorded at his own Belmont Mall Studio and produced by Mellencamp. The album is one of the most personal to date for the outspoken artist as he continues to show that he\'s one of the best songwriters of his generation, focusing on social issues with standout songs "Hey God" and "The Eyes of Portland."']}

In [82]:
reference_input=list(client.list_examples(dataset_id=dataset.id,limit=10))[9].inputs
reference_output=list(client.list_examples(dataset_id=dataset.id,limit=10))[9].outputs

### rag pipeline

In [83]:
from groq import Groq
from qdrant_client import QdrantClient
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json


def _mean_pool_embedding(raw_embedding):
    if not raw_embedding:
        raise ValueError("Hugging Face API returned an empty embedding.")

    if isinstance(raw_embedding[0], (int, float)):
        return [float(value) for value in raw_embedding]

    if isinstance(raw_embedding[0], list):
        token_count = len(raw_embedding)
        vector_size = len(raw_embedding[0])
        pooled = [0.0] * vector_size

        for token_vector in raw_embedding:
            if len(token_vector) != vector_size:
                raise ValueError("Inconsistent token vector dimensions in HF embedding response.")
            for idx, value in enumerate(token_vector):
                pooled[idx] += float(value)

        return [value / token_count for value in pooled]

    raise ValueError("Unexpected Hugging Face embedding response format.")

def get_embedding(text, model_name: str | None = None):
    """
    Embed text using HuggingFace Embeddings API.
    Model: sentence-transformers/all-MiniLM-L6-v2
    """
    selected_model = model_name or HF_EMBEDDING_MODEL
    endpoint = (
        "https://router.huggingface.co/hf-inference/models/"
        f"{selected_model}/pipeline/feature-extraction"
    )
    payload = json.dumps(
        {
            "inputs": text,
            "normalize": True,
        }
    ).encode("utf-8")

    headers = {"Content-Type": "application/json"}
    if HF_API_TOKEN:
        headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

    request = Request(endpoint, data=payload, headers=headers, method="POST")

    try:
        with urlopen(request, timeout=60) as response:
            response_data = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        message = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Hugging Face embedding API request failed ({exc.code}): {message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

    if isinstance(response_data, dict) and response_data.get("error"):
        raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

    if isinstance(response_data, list) and len(response_data) == 1:
        return _mean_pool_embedding(response_data[0])
    

    return _mean_pool_embedding(response_data)

def retrieve_data(query, qdrant_client, top_k=5):
    """
    Retrieve similar documents from Qdrant vector database.
    
    Args:
        query (str): User question
        qdrant_client: Qdrant client instance
        top_k (int): Number of documents to retrieve
    
    Returns:
        dict: Retrieved context with IDs, descriptions, ratings, and similarity scores
    """
    query_embedding = get_embedding(query)

    search_result = qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for search_result in search_result.points:
        retrieved_context_ids.append(search_result.payload["parent_asin"])
        retrieved_context.append(search_result.payload["description"])
        retrieved_context_ratings.append(search_result.payload["average_rating"])
        similarity_scores.append(search_result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

def process_context(context):
    """Format retrieved context for LLM prompt."""
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    """Build the prompt for the LLM."""
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in the stock.

    You will be given a question and a list of context.

    Instructions:
    -You need to answer the question based on the provided context only.
    -never use the word context and refer to it as the available products.

    Context:    
    {preprocessed_context}

    Question: {question}
""" 
    return prompt

# Initialize Groq client
client = Groq()

def generate_answer(prompt):
    """
    Generate answer using Groq LLM.
    
    Model: qwen/qwen3-32b
    
    Args:
        prompt (str): The formatted prompt with context and question
        
    Returns:
        str: Generated answer from the LLM
    """
    completion = client.chat.completions.create(
        model=GROQ_LLM_MODEL,  # qwen/qwen3-32b
        messages=[
        {
            "role": "system",
            "content": prompt
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden"
    )
    return completion.choices[0].message.content

def rag_pipeline(query, top_k=5):
    """
    Complete RAG Pipeline.
    
    Models Used:
    - Embedding: HuggingFace (sentence-transformers/all-MiniLM-L6-v2)
    - LLM: Groq (qwen/qwen3-32b)
    - Vector DB: Qdrant
    
    Args:
        query (str): User question
        top_k (int): Number of context documents to retrieve
        
    Returns:
        dict: {
            "answer": str - Generated answer,
            "question": str - Original question,
            "retrieved_context_ids": list - IDs of retrieved documents,
            "retrieved_context": list - Content of retrieved documents,
            "similarity_scores": list - Relevance scores
        }
    """
    qdrant_client = QdrantClient(url=QDRANT_URL)

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_result = {
        "answer": answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [84]:
rag_pipeline("What is the best album in terms of hard metal music?")

{'answer': 'Based on the provided products, **Soulfly’s *Totem* (ID: B09ZLQW9ZT)** and **Metallica’s *72 Seasons* (ID: B0BNJQMG9K)** both stand out as exceptional albums in the hard metal/brutal metal realm, with identical high ratings (4.7).  \n\n- **Soulfly’s *Totem*** is explicitly described as a "brutal, vibrant, extreme" release by Max Cavalera, featuring blackened thrash, death metal elements, and a raw, energetic sound honed over decades. It’s tailored for fans seeking aggressive, uncompromising metal.  \n- **Metallica’s *72 Seasons*** marks the legendary band’s first studio album since 2016, produced with their signature heavy sound and philosophical depth, though slightly more polished than extreme subgenres.  \n\nIf prioritizing *heaviness* and *extreme metal* energy, **Soulfly’s *Totem*** aligns more directly with the "hard metal" focus. However, for a *legendary metal band’s* new chapter, **Metallica’s *72 Seasons*** is the iconic choice. Both are highly recommended depend 

### ragas metrics

In [85]:

from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall,Faithfulness,ResponseRelevancy

C:\Users\Loq\AppData\Local\Temp\ipykernel_28052\3520975274.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall,Faithfulness,ResponseRelevancy
C:\Users\Loq\AppData\Local\Temp\ipykernel_28052\3520975274.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall,Faithfulness,ResponseRelevancy
C:\Users\Loq\AppData\Local\Temp\ipykernel_28052\3520975274.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metr

In [7]:
# Setup Gemini LLM and HuggingFace embeddings for RAGAS (no Groq / no langchain-groq)
# Ragas' LangchainLLMWrapper needs a LangChain model that supports async generation.

from langchain_google_genai import ChatGoogleGenerativeAI
import os


_gemini_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not _gemini_key:
    raise ValueError("Set GOOGLE_API_KEY (or GEMINI_API_KEY) in your .env to use Gemini for RAGAS.")

ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=_gemini_key,
    )
)
ragas_embeddings = LangchainEmbeddingsWrapper(hf)

C:\Users\Loq\AppData\Local\Temp\ipykernel_21168\3483175881.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
C:\Users\Loq\AppData\Local\Temp\ipykernel_21168\3483175881.py:19: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(hf)


In [87]:
reference_input

{'question': "What is the name of the album by John Mellencamp that features the song 'Orpheus Descending'?"}

In [88]:
reference_output

{'ground_truth': 'Orpheus Descending',
 'reference_context_ids': ['B0C2J9SYDQ'],
 'reference_descriptions': ['Orpheus Descending       Explicit Lyrics With a career spanning over 40 years, musician, artist and activist John Mellencamp is releasing his 25th full-length studio album, Orpheus Descending, via Republic Records. It was recorded at his own Belmont Mall Studio and produced by Mellencamp. The album is one of the most personal to date for the outspoken artist as he continues to show that he\'s one of the best songwriters of his generation, focusing on social issues with standout songs "Hey God" and "The Eyes of Portland."']}

In [89]:
result=rag_pipeline(reference_input["question"])

In [90]:
result

{'answer': 'The album by John Mellencamp that features the song "Orpheus Descending" is titled **Orpheus Descending**. It is his 25th full-length studio album, released via Republic Records.',
 'question': "What is the name of the album by John Mellencamp that features the song 'Orpheus Descending'?",
 'retrieved_context_ids': ['B0C2J9SYDQ',
  'B0BJP8ZBRT',
  'B09M57YCR2',
  'B09SJ3BCLJ',
  'B09Z6TY5F1'],
 'retrieved_context': ['Orpheus Descending       Explicit Lyrics With a career spanning over 40 years, musician, artist and activist John Mellencamp is releasing his 25th full-length studio album, Orpheus Descending, via Republic Records. It was recorded at his own Belmont Mall Studio and produced by Mellencamp. The album is one of the most personal to date for the outspoken artist as he continues to show that he\'s one of the best songwriters of his generation, focusing on social issues with standout songs "Hey God" and "The Eyes of Portland."',
  "MERCY . What does John Cale have th

In [91]:
async def ragas_faithfulness(run,example):
    sample=SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer=Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)

In [92]:
await ragas_faithfulness(result, "")

0.6666666666666666

In [93]:
async def ragas_response_relevancy(run, example):
    """
    Evaluate RAG output relevancy using RAGAS metrics.
    
    Models Used:
    - LLM: Groq (qwen/qwen3-32b)
    - Embeddings: HuggingFace (sentence-transformers/all-MiniLM-L6-v2)
    
    Args:
        run (dict): RAG pipeline output containing:
            - "question": str - The user query
            - "answer": str - Generated answer from RAG
            - "retrieved_context": list - Retrieved documents/context
            
        example (dict): Reference data from dataset (used for comparison)
    
    Returns:
        float: Relevancy score (0-1) indicating how relevant the answer is to the question
    """
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_context=run["retrieved_context"],
    )
    scorer = ResponseRelevancy(llm=ragas_llm,embeddings=ragas_embeddings)
    return await scorer.single_turn_ascore(sample)

In [94]:
await ragas_response_relevancy(result, "")

np.float64(0.9840324411830581)

In [99]:
async def ragas_context_precision_id_based(run, example):
    """
    Evaluate RAG context precision using RAGAS ID-Based Context Precision metric.
    
    Models Used:
    - LLM: Groq (qwen/qwen3-32b)
    - Embeddings: HuggingFace (sentence-transformers/all-MiniLM-L6-v2)
    
    Args:
        run (dict): RAG pipeline output containing:
            - "question": str - The user query
            - "answer": str - Generated answer from RAG
            - "retrieved_context_ids": list - IDs of retrieved documents
            
        example (dict): Reference data from dataset containing:
            - "relevant_context_ids": list - IDs of relevant documents for the question
    
    Returns:
        float: Precision score (0-1) indicating the proportion of retrieved context that is relevant
    """
    sample = SingleTurnSample(
        
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [100]:
await ragas_context_precision_id_based(result, reference_output)

0.2

In [101]:
async def ragas_context_recall_id_based(run, example):
    """
    Evaluate RAG context recall using RAGAS ID-Based Context Recall metric.
    
    Models Used:
    - LLM: Groq (qwen/qwen3-32b)
    - Embeddings: HuggingFace (sentence-transformers/all-MiniLM-L6-v2)
    
    Args:
        run (dict): RAG pipeline output containing:
            - "question": str - The user query
            - "answer": str - Generated answer from RAG
            - "retrieved_context_ids": list - IDs of retrieved documents
            
        example (dict): Reference data from dataset containing:
            - "relevant_context_ids": list - IDs of relevant documents for the question
    
    Returns:
        float: Recall score (0-1) indicating the proportion of relevant context that was retrieved
    """
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

In [102]:
await ragas_context_recall_id_based(result, reference_output)

1.0

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0,
    google_api_key=_gemini_key,
)

response = gemini.invoke("Hello, how are you?")
print(response.content)

[{'type': 'text', 'text': "Hello! I'm doing great, thank you for asking. How are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EjQKMgG+Pvb7wBF0m5bbArgGYC1mes2EdDXd2D86AD0fe6bdo/OiMwMExXmS27K93kyhcFgA'}}]
